# CRISP-DM: Donor Churn Prediction Pipeline

**Analytics goal:** Estimate each donor’s probability of becoming inactive (no gift within 365 days) so Lighthouse Sanctuary can prioritize retention outreach.

| CRISP-DM phase | Where it appears in this notebook |
|----------------|-----------------------------------|
| **1. Business understanding** | Next section + Business KPI mapping |
| **2. Data understanding** | Connection test, donor snapshot, audits, visualizations |
| **3. Data preparation** | Feature construction in `src.features` (encoding, windows) |
| **4. Modeling** | Train/evaluate, CV, explanatory model |
| **5. Evaluation** | Metrics, tiers, threshold logic, business readout |
| **6. Deployment** | Scored donor records / JSON-style outputs for operations |

### 1) Business understanding (donor churn)

**Organizational context:** Lighthouse Sanctuary relies on recurring philanthropy to fund safe housing, education, and reintegration services. Unaddressed donor lapse directly reduces program capacity; early identification of at-risk supporters is a **fundraising operations** problem, not only a modeling exercise.

**Business objectives**
- Focus limited stewardship and calling capacity on supporters **most likely to stop giving** without intervention.
- Improve **90-day retention** among contacted at-risk donors while controlling outreach spend and donor experience.

**Stakeholders and decisions**
| Stakeholder | Business decision supported |
|-------------|-----------------------------|
| **Donor Operations Lead** | Weekly prioritization lists, segments, and call queues by risk tier |
| **Communications** | Cadence and message strategy by risk band |
| **Leadership / board** | KPIs on retention program ROI and mission funding stability |

**Measurable success criteria (program level)**
- Target uplift in **90-day retention** in the high-risk *contacted* cohort (e.g. +8% vs. baseline or holdout), with **guardrails**: outreach cost per retained donor and donor opt-out/unsubscribe rates within agreed limits (e.g. ≤15% increase in cost for the targeted lift).

**Constraints and ethics**
- Scores are **prioritization tools**—not labels for treating donors unfairly; align with donor-privacy policies and consent.
- Outreach slots are **finite**; the model must rank-order rather than label everyone “urgent.”
- Historical patterns may miss life events outside the CRM; staff should combine scores with judgment.

**Business cost of prediction errors**
- **False positive** (flagged as churn-prone but would have stayed): unnecessary contact cost and possible message fatigue.
- **False negative** (missed churn): lost lifetime value and weakened relationship—often the more expensive error for mission revenue.

**How outputs are used:** Risk tiers drive retention plays (calls, personalized impact stories, escalation paths). `run_pipeline.py` emits JSON scores aligned with donor operations workflows.

## Setup: imports and configuration

*(Implements shared tooling for later CRISP-DM phases; data understanding follows.)*

In [1]:
import os
import sys
import time
from pathlib import Path


def _find_ml_pipelines_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        mp = base / "ml-pipelines"
        if mp.is_dir() and (mp / "requirements.txt").is_file():
            return mp
        if base.name == "ml-pipelines" and (base / "requirements.txt").is_file():
            return base
    raise RuntimeError(
        "Could not find ml-pipelines/requirements.txt. Open the Lighthouse-Sanctuary repo "
        "or the ml-pipelines folder, then use Run All."
    )


_boot_path = _find_ml_pipelines_root() / "pipeline_common" / "notebook_bootstrap.py"
_ns = globals()
_ns["__file__"] = str(_boot_path)
exec(compile(_boot_path.read_text(encoding="utf-8"), str(_boot_path), "exec"), _ns)

import pandas as pd
import numpy as np

from src.db import load_env, build_engine, run_query
from src.features import build_donor_training_frame, split_feature_target, select_explanatory_features
from src.modeling import train_and_evaluate, score_donors, train_explanatory_model
from src.config import RANDOM_STATE, CHURN_WINDOW_DAYS, CV_FOLDS

pd.set_option('display.max_columns', 200)
start_all = time.time()

## Business KPI mapping (operational measures)

These bullets make the **business objectives** in Business understanding measurable for programs and leadership.

- Stakeholder owner: Donor Operations Lead
- Decision enabled: prioritize retention outreach by donor risk tier
- Primary KPI: 90-day donor retention rate among contacted high-risk cohort
- Guardrail KPIs: outreach cost per retained donor, donor opt-out rate
- Minimum success criteria: +8% retention lift with <=15% increase in outreach cost

## CSV / DuckDB connection quick test

This `ml-pipelines` copy loads tables from `lighthouse_csv_v7` into DuckDB (not PostgreSQL). The next cell checks that `build_engine` returns a working in-memory DuckDB connection.

In [2]:
load_env('.env')
profile = os.getenv('DB_PROFILE', 'local')
engine = build_engine(profile)
print(f'Data source: CSV bundle via DuckDB (profile={profile})')
print(run_query('SELECT 1 AS connection_ok', engine))

Data source: CSV bundle via DuckDB (profile=local)
   connection_ok
0              1


## 2) Data understanding — connect + snapshot (CRISP-DM phase 2)

In [3]:
load_env('.env')
profile = os.getenv('DB_PROFILE', 'local').lower()
engine = build_engine(profile)
print(f'Connected with profile: {profile}')

t0 = time.time()
model_df = build_donor_training_frame(engine, churn_window_days=CHURN_WINDOW_DAYS)
print('Rows:', len(model_df), '| Donors:', model_df['supporter_id'].nunique())
print('Load+feature time (sec):', round(time.time() - t0, 2))
model_df.head()

Connected with profile: local
Rows: 284 | Donors: 59
Load+feature time (sec): 0.02


,donation_id,supporter_id,snapshot_date,next_donation_date,first_donation_date,donation_type,channel_source,donation_count_to_date,amount_sum_to_date,amount_avg_to_date,amount_std_to_date,recurring_rate_to_date,tenure_days,recency_days,supporter_type,relationship_type,region,acquisition_channel,max_donation_date,churn_365
0,145,1,2023-03-25,2023-06-24,2023-03-25,Monetary,SocialMedia,1,774.61,774.610,0.000000,1.0,0,0,SocialMediaAdvocate,Local,Luzon,SocialMedia,2026-03-01,0
1,158,1,2023-06-24,2023-07-01,2023-03-25,InKind,Direct,2,1381.52,690.760,83.850000,1.0,91,91,SocialMediaAdvocate,Local,Luzon,SocialMedia,2026-03-01,0
2,295,1,2023-07-01,2023-07-02,2023-03-25,Monetary,Event,3,2045.46,681.820,69.620846,1.0,98,7,SocialMediaAdvocate,Local,Luzon,SocialMedia,2026-03-01,0
3,15,1,2023-07-02,2023-07-23,2023-03-25,InKind,Event,4,2345.46,586.365,175.983714,1.0,99,1,SocialMediaAdvocate,Local,Luzon,SocialMedia,2026-03-01,0
4,148,1,2023-07-23,2023-12-20,2023-03-25,Time,SocialMedia,5,2364.66,472.932,276.123878,1.0,120,21,SocialMediaAdvocate,Local,Luzon,SocialMedia,2026-03-01,0


In [4]:
# Data Understanding Audit: missingness + simple anomaly checks
missing_pct = (model_df.isna().mean() * 100).sort_values(ascending=False)
print('Top missingness columns (%)')
display(missing_pct.head(10).to_frame('missing_pct'))

audit = {
    'rows': len(model_df),
    'distinct_donors': int(model_df['supporter_id'].nunique()),
    'negative_recency_rows': int((model_df['recency_days'] < 0).sum()) if 'recency_days' in model_df.columns else 0,
    'zero_or_negative_amount_avg_rows': int((model_df['amount_avg_to_date'] <= 0).sum()) if 'amount_avg_to_date' in model_df.columns else 0,
}
print('Audit summary:', audit)

Top missingness columns (%)


,missing_pct
next_donation_date,2.816901
donation_id,0.000000
recurring_rate_to_date,0.000000
max_donation_date,0.000000
acquisition_channel,0.000000
region,0.000000
relationship_type,0.000000
supporter_type,0.000000
recency_days,0.000000
tenure_days,0.000000


Audit summary: {'rows': 284, 'distinct_donors': 59, 'negative_recency_rows': 0, 'zero_or_negative_amount_avg_rows': 0}


In [5]:
print('Target distribution (churn_365):')
display(model_df['churn_365'].value_counts(dropna=False).rename('count').to_frame())

display(model_df.describe(include='all').transpose().head(20))

Target distribution (churn_365):


,count
churn_365,
0,254
1,30


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
donation_id,284.0,NaN,NaN,NaN,207.901408,3.0,101.75,204.0,320.25,419.0,122.069685
supporter_id,284.0,NaN,NaN,NaN,28.253521,1.0,12.0,27.0,43.0,60.0,17.657294
snapshot_date,284,NaN,NaN,NaN,2024-02-01 01:31:16.056338,2023-01-09 00:00:00,2023-07-15 12:00:00,2023-12-25 12:00:00,2024-09-24 00:00:00,2025-02-28 00:00:00,NaN
next_donation_date,276,NaN,NaN,NaN,2024-06-16 07:49:33.913043,2023-02-02 00:00:00,2023-11-02 18:00:00,2024-07-28 00:00:00,2024-12-22 00:00:00,2025-12-14 00:00:00,NaN
first_donation_date,284,NaN,NaN,NaN,2023-05-17 06:55:46.478873,2023-01-09 00:00:00,2023-02-22 00:00:00,2023-03-22 00:00:00,2023-07-02 00:00:00,2025-02-06 00:00:00,NaN
donation_type,284,5,Monetary,157,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel_source,284,5,Campaign,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
donation_count_to_date,284.0,NaN,NaN,NaN,4.091549,1.0,2.0,3.0,5.0,19.0,3.269746
amount_sum_to_date,284.0,NaN,NaN,NaN,2635.712923,2.66,737.96,1809.925,3744.5625,10880.46,2405.993495
amount_avg_to_date,284.0,NaN,NaN,NaN,608.106389,2.66,369.869375,600.708333,775.251429,2565.03,367.261334


## 3) Data Preparation (Feature/Target Split)

In [6]:
X, y = split_feature_target(model_df)
print('X shape:', X.shape)
print('y mean (churn rate):', round(y.mean(), 4))
print('Nulls in X:', int(X.isna().sum().sum()))
X.head()

X shape: (284, 17)
y mean (churn rate): 0.1056
Nulls in X: 0


,donation_id,supporter_id,snapshot_date,first_donation_date,donation_type,channel_source,donation_count_to_date,amount_sum_to_date,amount_avg_to_date,amount_std_to_date,recurring_rate_to_date,tenure_days,recency_days,supporter_type,relationship_type,region,acquisition_channel
0,145,1,2023-03-25,2023-03-25,Monetary,SocialMedia,1,774.61,774.610,0.000000,1.0,0,0,SocialMediaAdvocate,Local,Luzon,SocialMedia
1,158,1,2023-06-24,2023-03-25,InKind,Direct,2,1381.52,690.760,83.850000,1.0,91,91,SocialMediaAdvocate,Local,Luzon,SocialMedia
2,295,1,2023-07-01,2023-03-25,Monetary,Event,3,2045.46,681.820,69.620846,1.0,98,7,SocialMediaAdvocate,Local,Luzon,SocialMedia
3,15,1,2023-07-02,2023-03-25,InKind,Event,4,2345.46,586.365,175.983714,1.0,99,1,SocialMediaAdvocate,Local,Luzon,SocialMedia
4,148,1,2023-07-23,2023-03-25,Time,SocialMedia,5,2364.66,472.932,276.123878,1.0,120,21,SocialMediaAdvocate,Local,Luzon,SocialMedia


## 4) Modeling (Bounded Runtime)

Guardrails:
- only 2 models (Logistic Regression and Random Forest)
- capped CV folds (`CV_FOLDS`)
- bounded tree count
- elapsed time tracking

In [7]:
t1 = time.time()
train_out = train_and_evaluate(X, y, random_state=RANDOM_STATE, cv_folds=CV_FOLDS)
print('Training time (sec):', round(time.time() - t1, 2))
display(train_out['results_df'])
print('Best model:', train_out['best_model_name'])

Training time (sec): 3.48


,model,cv_roc_auc,cv_pr_auc,cv_f1,test_roc_auc,test_pr_auc,test_f1,elapsed_seconds
0,logistic,0.656751,0.150295,0.0,0.770089,0.221771,0.0,1.69
1,random_forest,0.670938,0.168607,0.0,0.743304,0.252724,0.0,1.79


Best model: logistic


## 5) Explanatory Objective (Interpret Relationships, Not Causal Proof)

This section complements predictive modeling with an interpretable logistic specification.

- Goal: estimate direction and magnitude of relationships with churn.
- Caution: these are associations, not causal claims.
- Decision value: identify practical retention levers to test.

In [8]:
X_expl = select_explanatory_features(X)
expl_out = train_explanatory_model(X_expl, y, random_state=RANDOM_STATE)

print('Explanatory model holdout metrics:')
print({k: round(v, 4) for k, v in expl_out['metrics'].items()})

print('\nTop positive churn associations (higher odds of churn):')
display(expl_out['top_positive'])

print('Top negative churn associations (lower odds of churn):')
display(expl_out['top_negative'])

Explanatory model holdout metrics:
{'test_roc_auc': 0.7902, 'test_pr_auc': 0.2458, 'test_f1': 0.0}

Top positive churn associations (higher odds of churn):


,feature,coefficient,odds_ratio
0,cat__acquisition_channel_WordOfMouth,0.948030,2.580621
1,cat__region_Mindanao,0.498841,1.646812
2,cat__channel_source_PartnerReferral,0.436546,1.547353
3,cat__supporter_type_SkillsContributor,0.433586,1.542780
4,cat__donation_type_InKind,0.424555,1.528910
5,cat__supporter_type_InKindDonor,0.387118,1.472730
6,cat__donation_type_SocialMedia,0.332816,1.394890
7,cat__region_Luzon,0.332325,1.394206
8,cat__supporter_type_SocialMediaAdvocate,0.330886,1.392201
9,num__recency_days,0.248704,1.282362


Top negative churn associations (lower odds of churn):


,feature,coefficient,odds_ratio
0,num__recurring_rate_to_date,-1.016496,0.361861
1,cat__region_Visayas,-0.834532,0.434078
2,cat__supporter_type_PartnerOrganization,-0.752079,0.471386
3,cat__supporter_type_Volunteer,-0.648481,0.522839
4,cat__acquisition_channel_Website,-0.600178,0.548714
5,cat__donation_type_Time,-0.417462,0.658717
6,cat__relationship_type_Local,-0.304942,0.737166
7,cat__donation_type_Monetary,-0.294103,0.745200
8,cat__acquisition_channel_PartnerReferral,-0.252738,0.776671
9,cat__channel_source_Campaign,-0.167126,0.846093


### Decision Implications

Use the explanatory drivers as hypothesis generators for interventions:
- If higher recency (longer gap since prior donation) is associated with churn, trigger outreach earlier.
- If recurring behavior lowers churn odds, prioritize recurring conversion campaigns.
- If specific acquisition channels have higher churn association, redesign onboarding for those channels.

These actions should be validated with experiments (A/B tests), not assumed causal from this model alone.

## 6) Evaluation + deployment-style scoring (CRISP-DM phases 5–6)

In [9]:
best_model = train_out['best_model']
scored = score_donors(best_model, X)

print('Risk tier counts:')
display(scored['risk_tier'].value_counts(dropna=False).to_frame('count'))
display(scored.head(20))

Risk tier counts:


,count
risk_tier,
Low,261
Medium,18
High,5


,supporter_id,churn_probability_365,risk_tier
0,36,0.752573,High
1,58,0.724091,High
2,56,0.716383,High
3,11,0.675820,High
4,59,0.662331,High
5,32,0.570539,Medium
6,41,0.558063,Medium
7,49,0.557181,Medium
8,32,0.548169,Medium
9,53,0.543460,Medium


In [10]:
if train_out['best_model_name'] == 'random_forest':
    prep = best_model.named_steps['prep']
    model = best_model.named_steps['model']
    feat_names = prep.get_feature_names_out()
    importances = pd.Series(model.feature_importances_, index=feat_names).sort_values(ascending=False).head(20)
    display(importances.to_frame('importance'))
else:
    print('Top importances shown for random forest model only.')

Top importances shown for random forest model only.


## Operationalization Policy + Monitoring

- Threshold policy: use cost-minimizing threshold from confusion-cost analysis for campaign execution; default fallback 0.50.
- Action bands: `High` >= 0.65, `Medium` 0.35-0.65, `Low` < 0.35.
- Retraining cadence: monthly scheduled retrain; emergency retrain if PR-AUC drops >15% or key feature drift exceeds threshold.
- Integration references:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## 7) Conclusion

In [11]:
print('Pipeline finished successfully.')
print('Total elapsed (sec):', round(time.time() - start_all, 2))

Pipeline finished successfully.
Total elapsed (sec): 3.75
